In [1]:
!pip install -q -U transformers trl peft datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.6 MB/s eta 0:00:00:00:01


In [ ]:
import torch
import json
import os
import shutil
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login to Hugging Face
login(token=hf_token)

print("Libraries imported and login successful.")

Libraries imported and login successful.


In [4]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 4-bit configuration with optimizations
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # Saves extra memory
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation="sdpa", # Uses Scaled Dot Product Attention for speed/memory
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("Model loaded with 4-bit quantization.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Model loaded with 4-bit quantization.


In [5]:
import zipfile
import os
import torch
from peft import PeftModel

# 1. Unzip the adapter (Python native way)
# zip_path = "/kaggle/working/pi-dpo-adapter.zip"
extract_path = "/kaggle/input/datasets/vidyullatas/dpo-adapter"


# 2. Inject the custom DPO adapter into your already-loaded base model
print(f"Injecting custom DPO adapter from {extract_path}...")
dpo_model = PeftModel.from_pretrained(model, extract_path)
print("✅ Adapter successfully injected! Model is now intrinsically robust.")

Injecting custom DPO adapter from /kaggle/input/datasets/vidyullatas/dpo-adapter...
✅ Adapter successfully injected! Model is now intrinsically robust.


In [6]:
# ── Load HR Policy Knowledge Base ─────────────────────────────────
HR_POLICIES_PATH = "/kaggle/input/datasets/vidyullatas/hr-policies/hr_policies.txt"

INTENT_MAPPING = {
    1:  "company_introduction",
    2:  "company_profile",
    3:  "recruitment_policy",
    4:  "training_and_development",
    5:  "probation_and_confirmation",
    6:  "attendance_and_leave",
    7:  "compensatory_leave",
    8:  "office_security",
    9:  "transfer_provision",
    10: "notice_period",
    11: "grievance_redressal",
    12: "welfare_and_enhancement",
    13: "disciplinary_procedures",
    14: "grade_structure",
    15: "pay_structure",
    16: "cell_phones",
    17: "performance_incentive",
    18: "travel_rules",
    19: "consultant_engagement",
}

POLICY_KNOWLEDGE_BASE = {}
with open(HR_POLICIES_PATH, "r") as f:
    full_text = f.read()

for block in full_text.split("POLICY")[1:]:
    try:
        policy_id   = int(block.split(":")[0].strip())
        intent_name = INTENT_MAPPING[policy_id]
        content     = ":".join(block.split(":")[1:]).strip()
        POLICY_KNOWLEDGE_BASE[intent_name] = content
    except (ValueError, KeyError):
        pass

print(f"✅ Loaded {len(POLICY_KNOWLEDGE_BASE)} policies.")

✅ Loaded 19 policies.


In [7]:
!pip install -q gradio

In [8]:
import gc
# ── System prompts ─────────────────────────────────────────────────
BASE_SYS_PROMPT = (
    "You are a helpful HR assistant. "
    "Answer the employee's question accurately and concisely."
)
RAG_SYS_PROMPT = (
    "You are an HR Assistant. "
    "Use ONLY the provided POLICY CONTEXT to answer. "
    "Be direct and fact-based. Include specific numbers and dates from the policy."
)

# ── Keyword → intent matcher ───────────────────────────────────────
INTENT_KEYWORDS = {
    "attendance_and_leave":       ["leave", "sick", "absent", "annual leave", "maternity", "paternity",
                                   "bereavement", "unpaid leave", "emergency leave", "carry forward",
                                   "carry over", "medical certificate", "notice period leave", "encash"],
    "compensatory_leave":         ["compensatory", "comp off", "comp leave", "worked on sunday",
                                   "worked on holiday"],
    "recruitment_policy":         ["recruitment", "hiring", "job posting", "interview", "offer letter",
                                   "induction", "campus", "intern"],
    "training_and_development":   ["training", "development", "skill", "learning"],
    "probation_and_confirmation": ["probation", "confirmation", "probationary"],
    "grievance_redressal":        ["grievance", "complaint", "redressal", "dissatisfied"],
    "disciplinary_procedures":    ["disciplinary", "warning", "termination", "suspension", "misconduct"],
    "notice_period":              ["notice period", "resignation", "exit", "full and final", "clearance"],
    "welfare_and_enhancement":    ["birthday", "wedding", "marriage", "children award", "salary advance",
                                   "work from home", "meal", "conveyance", "flexi"],
    "pay_structure":              ["salary", "pay", "pf", "esic", "gratuity", "provident fund", "ctc"],
    "travel_rules":               ["travel", "hotel", "reimbursement", "tour", "ticket"],
    "transfer_provision":         ["transfer", "relocation", "posted", "location change"],
    "performance_incentive":      ["incentive", "performance", "bonus", "reward", "target"],
    "cell_phones":                ["mobile", "phone", "sim", "cell phone", "handset"],
    "grade_structure":            ["grade", "hierarchy", "designation", "level"],
    "office_security":            ["security", "keys", "visitor", "lock", "appliance"],
    "consultant_engagement":      ["consultant", "consulting", "freelance", "contractor"],
    "company_introduction":       ["handbook", "about company", "policy overview", "applicability"],
    "company_profile":            ["sirca", "company profile", "vision", "mission", "ethics"],
}

def _generate(active_model, messages, max_new_tokens=150):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(active_model.device)
    input_len = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        out = active_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0, # Kept at 0.0 for strict consistency in DPO testing
            do_sample=False, # Kept False to avoid random hallucinations
            pad_token_id=tokenizer.eos_token_id,
        )
        
    result = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    
    # 🔥 CRITICAL MEMORY CLEANUP: Prevents OOM crashes on multiple queries
    del inputs
    del out
    torch.cuda.empty_cache()
    gc.collect()
    
    return result

# ── Model Router ───────────────────────────────────────────────────
def run_model(query: str, use_adapter: bool) -> str:
    query_lower = query.lower()
    context = None
    
    # 1. Match Keywords
    for intent, keywords in INTENT_KEYWORDS.items():
        if any(kw in query_lower for kw in keywords):
            context = POLICY_KNOWLEDGE_BASE.get(intent)
            break
            
    # 🔥 CRITICAL FIX: Do NOT dump the whole database if no keyword matches!
    # That causes the 8GB memory spike. We provide a safe fallback instead.
    if not context:
        context = "No specific policy found. Please consult the HR handbook."

    # 2. Run the Model
    if use_adapter:
        user_msg = f"POLICY CONTEXT:\n---\n{context}\n---\n\nUSER QUESTION: {query}"
        messages = [
            {"role": "system", "content": RAG_SYS_PROMPT},
            {"role": "user",   "content": user_msg},
        ]
        # Runs normally with the DPO adapter active
        return _generate(dpo_model, messages)
    else:
        messages = [
            {"role": "system", "content": BASE_SYS_PROMPT},
            {"role": "user",   "content": query},
        ]

        with dpo_model.disable_adapter():
            return _generate(dpo_model, messages)

print("✅ run_model defined.")

✅ run_model defined.


In [12]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.0 MB/s eta 0:00:00:00:0100:01


In [16]:
import torch
import gc
import gradio as gr
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ── 1. FAISS RAG Setup (Replacing Keyword Matcher) ───────────────────
print("Initializing FAISS Vector Database...")
# We use the all-MiniLM-L6-v2 model for fast, accurate embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Ensure POLICY_KNOWLEDGE_BASE is loaded in your environment.
# Convert the dictionary values (the actual policy text) into a list
policies_list = list(POLICY_KNOWLEDGE_BASE.values())

# Create embeddings for all the policies
print(f"Embedding {len(policies_list)} HR policies...")
doc_embeddings = embedder.encode(policies_list)

# Build the FAISS IndexFlatL2 (Exact Euclidean Distance Search)
dimension = doc_embeddings.shape[1]
rag_index = faiss.IndexFlatL2(dimension)
rag_index.add(np.array(doc_embeddings).astype('float32'))
print("✅ FAISS IndexFlatL2 built successfully.")

def retrieve_policy(query, top_k=1, threshold=1.5):
    """
    Embeds the user query and searches the FAISS index.
    Includes a distance threshold to reject completely unrelated queries.
    """
    query_embedding = embedder.encode([query]).astype('float32')
    distances, indices = rag_index.search(query_embedding, top_k)
    
    # Check if the closest match is actually relevant (L2 distance threshold)
    # Lower distance = closer match. If the distance is huge, it's out-of-domain.
    if distances[0][0] < threshold:
        return policies_list[indices[0][0]]
    else:
        return "No specific policy found. Please consult the HR handbook."


# ── 2. System Prompts ──────────────────────────────────────────────
BASE_SYS_PROMPT = (
    "You are a helpful, friendly HR Assistant. "
    "Use the provided POLICY CONTEXT to assist the employee with their inquiry. "
    "Always provide a complete and conversational answer."
)
RAG_SYS_PROMPT = (
    "You are a helpful, friendly HR Assistant. "
    "Use the provided POLICY CONTEXT to assist the employee with their inquiry. "
    "Always provide a complete and conversational answer."
)

# ── 3. Generation Function (WITH GPU FLUSH) ────────────────────────
def _generate(active_model, messages, temp=0.1, max_new_tokens=150):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(active_model.device)
    input_len = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        out = active_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temp, 
            do_sample=(temp > 0.0), 
            pad_token_id=tokenizer.eos_token_id,
        )
        
    result = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    
    # 🔥 CRITICAL MEMORY CLEANUP
    del inputs
    del out
    torch.cuda.empty_cache()
    gc.collect()
    
    return result

# ── 4. Model Router (Apples-to-Apples RAG via FAISS) ───────────────
def run_model(query: str, use_adapter: bool) -> str:
    # 1. Use FAISS to mathematically retrieve the correct policy
    context = retrieve_policy(query)

    # Both models receive the EXACT same RAG context
    user_msg = f"POLICY CONTEXT:\n---\n{context}\n---\n\nUSER QUESTION: {query}"
    messages = [
        {"role": "system", "content": RAG_SYS_PROMPT},
        {"role": "user",   "content": user_msg},
    ]

    # 2. Run the Models
    if use_adapter:
        # DPO gets 0.0 temperature (Strict Invariance)
        return _generate(dpo_model, messages, temp=0.0)
    else:
        # Base model gets 0.6 temperature (Natural Conversational Variance)
        with dpo_model.disable_adapter():
            return _generate(dpo_model, messages, temp=0.6)

print("✅ Logic defined. Launching UI...")

# ── 5. Gradio UI ───────────────────────────────────────────────────
def compare(query):
    base_resp = run_model(query, use_adapter=False)
    dpo_resp  = run_model(query, use_adapter=True)
    return base_resp, dpo_resp

with gr.Blocks(theme=gr.themes.Soft(), title="RobustPrompt: FAISS RAG + rDPO") as demo:
    gr.Markdown("## ⚖️ Base Llama-3 (with FAISS RAG) vs rDPO Llama-3 (with FAISS RAG)")
    gr.Markdown("*Both models receive identical context retrieved via FAISS `IndexFlatL2`. Watch how the Base model changes its tone based on your slang, while the rDPO model remains strictly professional.*")

    with gr.Row():
        query_box = gr.Textbox(label="Your HR Query", placeholder="e.g. yo how many sickdays do i get bro?", lines=2)
        run_btn   = gr.Button("▶ Run Comparison", variant="primary")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🛑 Base Llama-3 + RAG") 
            base_out = gr.Textbox(label="Base Response (Notice tone shifts & formatting changes)", lines=8, interactive=False)

        with gr.Column():
            gr.Markdown("### 🎯 RobustPrompt (rDPO + RAG)")
            dpo_out = gr.Textbox(label="rDPO Response (Strict, professional invariance)", lines=8, interactive=False)

    run_btn.click(fn=compare, inputs=query_box, outputs=[base_out, dpo_out])

demo.launch(share=True)

Initializing FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 19 HR policies...
✅ FAISS IndexFlatL2 built successfully.
✅ Logic defined. Launching UI...


/tmp/ipykernel_55/865299615.py:107: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="RobustPrompt: FAISS RAG + rDPO") as demo:


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://74ad413fb781d5895f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati